In [1]:
import numpy as np
from pykevlar.grid import HyperPlane
from pykevlar.model.binomial import SimpleSelection
from pykevlar.core.model import mt19937
from pykevlar.driver import accumulate_process
from utils import make_cartesian_grid_range

n_arms = 2
n_thetas = 8
sim_size = 200
seed = 10

model = SimpleSelection(n_arms, 35, 20, [-0.0])

gr = make_cartesian_grid_range(n_thetas, np.full(n_arms, -0.5), np.full(n_arms, 0.5), 1000)

# define null hypos
null_hypos = []
for i in range(1, n_arms):
    n = np.zeros(n_arms)
    n[0] = 1
    n[i] = -1
    null_hypos.append(HyperPlane(n, 0))

out = accumulate_process(model, gr, sim_size, seed, 1)

In [ ]:
out.typeI_sum()

In [ ]:
sgs = model.make_sim_global_state(gr)
ss = sgs.make_sim_state()
gen = mt19937(seed)
pos_start = np.cumsum(gr.n_tiles_per_pt, dtype=np.int64) - gr.n_tiles_per_pt[0]
typeI_sum = np.zeros(gr.n_tiles())
score_sum = np.zeros((gr.n_tiles(), 2))
# We can run in blocks of 1000 sim_size. Adagrid can just discretize in blocks
# of that size. Then, the set of grid points that need to be run over simulation
# block.
rej_len = np.zeros((sim_size, gr.n_tiles()), dtype=np.uint32)
score = np.empty((sim_size, gr.n_gridpts(), n_arms))
for i in range(sim_size):
    ss.simulate(gen, rej_len[i])
    rej_len_per_gridpt = np.add.reduceat(rej_len, pos_start)
    any_rejs = rej_len_per_gridpt > 0
    rej_idxs = np.where(any_rejs)[0]
    for j in range(gr.n_gridpts()):
        score_buf = np.empty(n_arms)
        ss.score(j, score_buf)
        score[j] = score_buf
    # ss.score(rej_idxs, score_buf)
    typeI_sum += rej_len
    score_sum[any_rejs] += score
    score_sum += score * (rej_len != 0)[:, None]
np.testing.assert_allclose(typeI_sum, out.typeI_sum()[0])
# np.testing.assert_allclose(score_sum, out.score_sum().reshape((-1, 2)))

ValueError: non-broadcastable output operand with shape (136,) doesn't match the broadcast shape (1000,136)